# 01 — Build an `EvaluationDataset` and round-trip it

The schema layer: constructing test cases in code, serializing to pandas / Excel,
and loading back with a column mapping.

**Runs fully offline** — no credentials, no heavy dependencies.

## 1. Construct test cases directly in code

In [ ]:
from llminspector.dataset import EvaluationDataset
from llminspector.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What is the capital of France?",
        actual_output="The capital of France is Paris.",
        expected_output="Paris",
        retrieval_context=["France is a country in Europe. Its capital is Paris."],
    ),
    LLMTestCase(
        input="Summarize the refund policy.",
        actual_output="Refunds are available within 30 days.",
        expected_output="Customers may request a refund within 30 days of purchase.",
    ),
]
dataset = EvaluationDataset(test_cases=test_cases)
dataset

## 2. Serialize to a DataFrame

Columns map back to the legacy names by default: `question` / `answer` /
`ground_truth` / `contexts` / `policy`.

In [ ]:
df = dataset.to_pandas()
df[["question", "answer", "ground_truth"]]

## 3. Round-trip through Excel

In [ ]:
dataset.to_excel("/tmp/llminspector_dataset.xlsx")

reloaded = EvaluationDataset.from_excel("/tmp/llminspector_dataset.xlsx")
print(reloaded)
reloaded.to_pandas().head()

## 4. Load with a custom column mapping

If your sheet uses different headers, name them on the way in. This is the
**read** schema — separate from the write schema used when exporting results.

In [ ]:
# reloaded = EvaluationDataset.from_excel(
#     "my_sheet.xlsx", input_col="prompt", actual_output_col="response",
# )